# PNG Gating for Text Transformers — BERT-base (Unfrozen Backbone)
## Offensive Language Detection · TweetEval / OffensEval (SemEval 2019)

Adapts **Patch-Norm Gating (PNG)** from Vision Transformers to **BERT-base-uncased**
for binary offensive language classification.

**Key differences vs frozen-backbone notebook:**
- Model: `bert-base-uncased` (110 M params, **12 layers**, 12 heads)
- Backbone **unfrozen** — full fine-tuning: all params train (gate params + encoder + pooler + head)
- Lower LR (2e-5) to avoid catastrophic forgetting
- Fewer epochs (5) — full fine-tuning converges faster

| | |
|---|---|
| **Model** | `bert-base-uncased` (110 M params, 12 layers, 12 heads) |
| **Dataset** | TweetEval/offensive · `cardiffnlp/tweet_eval` (SemEval 2019 Task 6) |
| **Task** | Binary: `not-offensive` (0) / `offensive` (1) |
| **Backbone** | Unfrozen — full fine-tuning |
| **Novel** | Token-norm-aware gating (PNG) adapted from ViT patch gating |

## 0. Setup

In [ ]:
import os, types, math, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
)
from datasets import load_dataset
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    accuracy_score, classification_report,
)
import pandas as pd

DEVICE = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MAX_LEN  = 128
BATCH    = 32
EPOCHS   = 5           # fewer epochs: full fine-tuning converges faster
LR_CLS   = 2e-5        # lower LR: avoid catastrophic forgetting
NUM_CLS  = 2           # TweetEval/offensive: 0=not-offensive  1=offensive
CKPT_DIR = Path('checkpoints_bert_unfrozen')
CKPT_DIR.mkdir(exist_ok=True)

def _safe_name(n):
    return n.replace(' ', '_').replace('-', '').replace('(', '').replace(')', '')

print(f'torch {torch.__version__} | device {DEVICE}')
print(f'Checkpoints -> {CKPT_DIR.resolve()}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Checkpoint restore — copy from previous Kaggle version output if available
#
# On Kaggle: attach the output of the previous notebook version as an input
# dataset (Edit -> Add data -> Your work -> this notebook -> version N).
# The files will appear under /kaggle/input/<notebook-slug>/checkpoints_bert_unfrozen/
# This cell finds them automatically and copies to the current working dir.
# ─────────────────────────────────────────────────────────────────────────────
import shutil, glob as _glob

CKPT_DIR = Path('checkpoints_bert_unfrozen')
CKPT_DIR.mkdir(exist_ok=True)

# Search all mounted input datasets for a checkpoints_bert_unfrozen folder
_src_dirs = _glob.glob('/kaggle/input/**/checkpoints_bert_unfrozen', recursive=True)

if _src_dirs:
    _src = Path(sorted(_src_dirs)[-1])  # pick latest if multiple
    print(f'Found checkpoint source: {_src}')
    _copied = 0
    for _f in _src.iterdir():
        _dst = CKPT_DIR / _f.name
        if not _dst.exists():          # don't overwrite locally produced files
            shutil.copy2(_f, _dst)
            print(f'  Copied: {_f.name}')
            _copied += 1
        else:
            print(f'  Skipped (exists): {_f.name}')
    print(f'Done — {_copied} file(s) copied.')
else:
    print('No previous checkpoint directory found under /kaggle/input/.')
    print('To restore checkpoints: Edit -> Add data -> Your work -> this notebook -> version N')

print(f'\nCheckpoints in working dir:')
for _f in sorted(CKPT_DIR.iterdir()):
    print(f'  {_f.name}  ({_f.stat().st_size/1e6:.1f} MB)')

## 1. Dataset & Tokenizer

In [ ]:
# Dataset: TweetEval / OffensEval (SemEval 2019 Task 6)

# ── Tokenizer ─────────────────────────────────────────────────────────────────
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# ── Load dataset ──────────────────────────────────────────────────────────────
raw = load_dataset('cardiffnlp/tweet_eval', 'offensive')
print(raw)
print('Sample:', raw['train'][0])

LABEL_NAMES = ['not-offensive', 'offensive']

# ── Tokenise ──────────────────────────────────────────────────────────────────
def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LEN,
        padding='max_length',
    )

tokenized = raw.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized.set_format('torch')

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_dl = DataLoader(tokenized['train'],      batch_size=BATCH, shuffle=True,  num_workers=0)
val_dl   = DataLoader(tokenized['validation'], batch_size=BATCH, shuffle=False, num_workers=0)
test_dl  = DataLoader(tokenized['test'],       batch_size=BATCH, shuffle=False, num_workers=0)

n_train = len(tokenized['train'])
n_val   = len(tokenized['validation'])
n_test  = len(tokenized['test'])
print(f'Train: {n_train} | Val: {n_val} | Test: {n_test}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Dataset Quality Analysis
# ─────────────────────────────────────────────────────────────────────────────

print('Dataset  : TweetEval / OffensEval  (SemEval 2019 Task 6)')
print('Source   : English Twitter  |  Annotations: OLID scheme  |  Licence: research use')
print()

for split in ('train', 'validation', 'test'):
    n      = len(raw[split])
    labels = [ex['label'] for ex in raw[split]]
    n_off  = labels.count(1)
    n_not  = labels.count(0)
    print(f'{split.upper()} ({n} samples):')
    print(f'  not-offensive : {n_not:5d}  ({100*n_not/n:.1f}%)  {"#"*int(50*n_not/n)}')
    print(f'  offensive     : {n_off:5d}  ({100*n_off/n:.1f}%)  {"#"*int(50*n_off/n)}')
    print(f'  Majority-class baseline: {max(n_not, n_off)/n:.4f}')
    print()

texts   = [ex['text'] for ex in raw['train']]
lengths = [len(t.split()) for t in texts]
n_trunc = sum(l > MAX_LEN for l in lengths)
med     = sorted(lengths)[len(lengths)//2]
print(f'Text length (train):')
print(f'  Mean: {sum(lengths)/len(lengths):.1f}  |  Median: {med}  |  Max: {max(lengths)}')
print(f'  Truncated (>{MAX_LEN} tokens): {n_trunc}/{len(lengths)}  ({100*n_trunc/len(lengths):.1f}%)')
print()
print('Inter-annotator agreement: OLID uses a tiered annotation scheme.')
print('Reported Cohen kappa in source paper (Zampieri et al. 2019): 0.73 (substantial).')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig 0 — Dataset Statistics
# ─────────────────────────────────────────────────────────────────────────────
cls_palette = ['#4e79a7', '#e15759']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('TweetEval/Offensive — Dataset Statistics', fontsize=13, fontweight='bold')

for ax, split in zip(axes[:2], ('train', 'test')):
    labels = [ex['label'] for ex in raw[split]]
    counts = [labels.count(i) for i in range(len(LABEL_NAMES))]
    bars   = ax.bar(LABEL_NAMES, counts, color=cls_palette, edgecolor='white', width=0.5)
    for bar, c in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 15, str(c),
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(f'{split.capitalize()}  (n={len(raw[split])})')
    ax.set_ylabel('Samples')

ax = axes[2]
texts   = [ex['text'] for ex in raw['train']]
lengths = [len(t.split()) for t in texts]
ax.hist(lengths, bins=40, color='#59a14f', edgecolor='white')
ax.axvline(MAX_LEN, color='red', linestyle='--', linewidth=1.5,
           label=f'MAX_LEN={MAX_LEN}')
ax.set_xlabel('Word count per tweet')
ax.set_ylabel('Frequency')
ax.set_title('Token-Length Distribution (train)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_0_dataset_stats.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_0_dataset_stats.pdf')

## 2. Gate Modules — G1 through G5 + PNG

### Gate positions (applied inside `BertSelfAttention`)
| Name | Where | Formula |
|------|--------|--------|
| **G1** | Attention output (per head, before out projection) | `context *= G` |
| **G2** | Value vectors | `v *= G` |
| **G3** | Key vectors | `k *= G` |
| **G4** | Query vectors | `q *= G` |
| **G5** | Final context (after reshape, before BertSelfOutput) | `context *= G` |
| **PNG** | G1 + token-norm correction | `G = sigmoid(h@W - β·||h||·e)` |

In [ ]:
class GateParams(nn.Module):
    # Per-head gating parameters for one BERT self-attention layer.
    #   W    : (hidden_size, num_heads) -- maps hidden state to gate logit per head
    #   e    : (num_heads,)             -- per-head norm scale  [PNG only]
    #   beta : (1,)                     -- global norm weight    [PNG only]
    def __init__(self, dim, num_heads, png=False):
        super().__init__()
        self.png = png
        self.W   = nn.Parameter(torch.empty(dim, num_heads))
        nn.init.normal_(self.W, std=0.02)
        if png:
            self.e    = nn.Parameter(torch.ones(num_heads))
            self.beta = nn.Parameter(torch.zeros(1))

    def gate(self, x):
        # x : (B, N, dim) -> G : (B, N, num_heads)
        g = x @ self.W
        if self.png:
            x_norm = x.norm(dim=-1, keepdim=True) + 1e-6   # (B, N, 1)
            g = g - self.beta * x_norm * self.e             # broadcast over heads
        return torch.sigmoid(g)

In [ ]:
def _make_gate_forward(module, params, pos):
    # Returns a patched forward method for BERT BertSelfAttention.

    def patched_forward(self, hidden_states, attention_mask=None,
                        head_mask=None, encoder_hidden_states=None,
                        encoder_attention_mask=None, past_key_value=None,
                        output_attentions=False, **kwargs):
        bs, q_len, _ = hidden_states.size()
        H  = self.num_attention_heads
        dh = self.attention_head_size

        # ── Gate from input hidden states ──────────────────────────────────────
        G   = params.gate(hidden_states)         # (B, N, H)
        G_h = G.permute(0, 2, 1).unsqueeze(-1)  # (B, H, N, 1)

        # ── Q / K / V projections → (B, H, N, dh) ─────────────────────────────
        def _t(x, length):
            return x.view(bs, length, H, dh).permute(0, 2, 1, 3)

        q = _t(self.query(hidden_states), q_len)
        k = _t(self.key(hidden_states),   q_len)
        v = _t(self.value(hidden_states), q_len)

        if pos == 'G4': q = q * G_h
        if pos == 'G3': k = k * G_h
        if pos == 'G2': v = v * G_h

        # ── Scaled dot-product attention ────────────────────────────────────────
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(dh)
        if attention_mask is not None:
            scores = scores + attention_mask
        attn_probs = F.softmax(scores, dim=-1)
        attn_probs = self.dropout(attn_probs)
        if head_mask is not None:
            attn_probs = attn_probs * head_mask

        context = torch.matmul(attn_probs, v)  # (B, H, N, dh)

        if pos in ('G1', 'PNG'):
            context = context * G_h

        # ── Reshape to (B, N, all_head_size) ────────────────────────────────────
        context = context.permute(0, 2, 1, 3).contiguous()
        context = context.view(bs, q_len, self.all_head_size)

        if pos == 'G5':
            context = (
                context.view(bs, q_len, H, dh) * G.unsqueeze(-1)
            ).view(bs, q_len, self.all_head_size)

        # ── Capture flags for visualisation ─────────────────────────────────────
        if getattr(self, '_capture_gate', False):
            self._last_gate   = G.detach().cpu()
            self._last_x_norm = hidden_states.norm(dim=-1).detach().cpu()
        if getattr(self, '_capture_attn', False):
            self._last_attn = attn_probs.detach().cpu()

        return (context, attn_probs)

    return types.MethodType(patched_forward, module)

In [ ]:
def _is_bert_self_attn(mod):
    return all(hasattr(mod, a) for a in
               ['query', 'key', 'value', 'num_attention_heads', 'attention_head_size', 'dropout'])


def inject_gates(model, pos, png=False):
    # Monkey-patch every BertSelfAttention module with a gate.
    # Returns nn.ModuleList of GateParams (one per attention layer).
    dim = model.config.hidden_size
    H   = model.config.num_attention_heads
    gate_list = []

    for name, mod in model.named_modules():
        if _is_bert_self_attn(mod):
            p = GateParams(dim, H, png=png)
            p = p.to(next(mod.parameters()).device)
            mod.forward = _make_gate_forward(mod, p, pos)
            gate_list.append(p)
            print(f'  Injected {pos} gate into {name}')

    gate_params = nn.ModuleList(gate_list)
    model.gate_params = gate_params
    return gate_params

## 3. Model & Training Helpers

In [ ]:
def build_model(pos='baseline', png=False, num_classes=NUM_CLS):
    # Load BERT-base with unfrozen backbone — full fine-tuning.
    # All parameters train (backbone + gate_params + classifier + pooler).
    m = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        num_labels=num_classes,
        ignore_mismatched_sizes=True,
    )

    if pos != 'baseline':
        print(f'Injecting {pos} gates (PNG={png}):')
        inject_gates(m, pos=pos, png=png)

    m = m.to(DEVICE)

    # ── All parameters trainable (unfrozen backbone) ───────────────────────────
    for p in m.parameters():
        p.requires_grad = True

    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in m.parameters())
    print(f'Trainable: {trainable:,} / {total:,}  (all unfrozen)')
    return m

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0.0
    criterion = nn.CrossEntropyLoss()

    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        attn = batch['attention_mask'].to(DEVICE)
        lbls = batch['label'].to(DEVICE)
        out  = model(input_ids=ids, attention_mask=attn)
        loss = criterion(out.logits, lbls)
        total_loss += loss.item()
        preds = out.logits.argmax(-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(lbls.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc  = accuracy_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    return dict(loss=avg_loss, acc=acc, f1=f1, precision=prec, recall=rec)

In [ ]:
def train(model, epochs=EPOCHS, lr=LR_CLS, name='model', resume_ckpt=None):
    # Full fine-tuning: all params train.
    # resume_ckpt: path to an epoch-level checkpoint to resume from.
    opt_params = [p for p in model.parameters() if p.requires_grad]
    optimizer  = torch.optim.AdamW(opt_params, lr=lr, weight_decay=0.01)
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs)
    criterion  = nn.CrossEntropyLoss()

    log = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
    start_epoch = 0

    # ── Resume from epoch-level checkpoint if available ────────────────────
    if resume_ckpt is not None and Path(resume_ckpt).exists():
        ckpt = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        log         = ckpt['log']
        start_epoch = ckpt['epoch']
        print(f'  Resumed from epoch {start_epoch} ({resume_ckpt})')

    for epoch in range(start_epoch, epochs):
        model.train()
        epoch_loss = 0.0
        for batch in tqdm(train_dl, desc=f'{name} E{epoch+1}/{epochs}', leave=False):
            ids  = batch['input_ids'].to(DEVICE)
            attn = batch['attention_mask'].to(DEVICE)
            lbls = batch['label'].to(DEVICE)
            optimizer.zero_grad()
            out  = model(input_ids=ids, attention_mask=attn)
            loss = criterion(out.logits, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(opt_params, 1.0)
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        train_loss  = epoch_loss / len(train_dl)
        val_metrics = evaluate(model, val_dl)
        val_acc  = val_metrics['acc']
        val_f1   = val_metrics['f1']
        val_loss = val_metrics['loss']
        log['train_loss'].append(train_loss)
        log['val_loss'].append(val_loss)
        log['val_acc'].append(val_acc)
        log['val_f1'].append(val_f1)
        print(f'  Ep {epoch+1}: train_loss={train_loss:.4f}'
              f' | val_acc={val_acc:.4f} | val_f1={val_f1:.4f}')

        # ── Epoch-level checkpoint ─────────────────────────────────────────
        sname_epoch = _safe_name(name)
        epoch_ckpt  = CKPT_DIR / f'{sname_epoch}_epoch{epoch+1}.pth'
        torch.save({
            'epoch':     epoch + 1,
            'model':     model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'log':       log,
        }, epoch_ckpt)
        if epoch > start_epoch:
            prev = CKPT_DIR / f'{sname_epoch}_epoch{epoch}.pth'
            if prev.exists():
                prev.unlink()

    last_epoch_ckpt = CKPT_DIR / f'{_safe_name(name)}_epoch{epochs}.pth'
    if last_epoch_ckpt.exists():
        last_epoch_ckpt.unlink()

    return log

## 4. Experiment Configuration & Ablation Run

Edit `EXPERIMENTS` below to control exactly what runs.  
Each entry is a plain dict — comment out any row to skip it.  
The run loop **auto-skips** experiments whose `.pth` + `results.json` already exist,
so re-running the cell after a crash picks up where it left off.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Experiment Configuration
# ─────────────────────────────────────────────────────────────────────────────
EXPERIMENTS = [
    dict(name='Baseline',          gate='baseline', epochs=EPOCHS, is_png=False, lr=LR_CLS),
    dict(name='G1 - Attn Output',  gate='G1',       epochs=EPOCHS, is_png=False, lr=LR_CLS),
    dict(name='G2 - Value Gate',   gate='G2',       epochs=EPOCHS, is_png=False, lr=LR_CLS),
    dict(name='G3 - Key Gate',     gate='G3',       epochs=EPOCHS, is_png=False, lr=LR_CLS),
    dict(name='G4 - Query Gate',   gate='G4',       epochs=EPOCHS, is_png=False, lr=LR_CLS),
    dict(name='G5 - Final Output', gate='G5',       epochs=EPOCHS, is_png=False, lr=LR_CLS),
    dict(name='PNG (novel)',        gate='PNG',      epochs=EPOCHS, is_png=True,  lr=LR_CLS),
]

print(f'{len(EXPERIMENTS)} experiments configured:')
header = f'{"Name":<26} {"Gate":<10} {"Epochs":>6}  {"PNG":<5}  {"LR"}'
print(header)
print('-' * len(header))
for exp in EXPERIMENTS:
    print(f'{exp["name"]:<26} {exp["gate"]:<10} {exp["epochs"]:>6}  {str(exp["is_png"]):<5}  {exp["lr"]}')

In [ ]:
import json as _json
import traceback as _tb
import glob as _glob

RESULTS_FILE = CKPT_DIR / 'results.json'

# Resume: load previously saved results
if RESULTS_FILE.exists():
    with open(RESULTS_FILE) as _f:
        results = _json.load(_f)
    print(f'Resumed {len(results)} result(s) from {RESULTS_FILE}')
else:
    results = {}

train_logs = {}

for exp in EXPERIMENTS:
    vname  = exp['name']
    pos    = exp['gate']
    png    = exp['is_png']
    epochs = exp['epochs']
    lr     = exp['lr']
    sname  = _safe_name(vname)
    ckpt_file = CKPT_DIR / f'{sname}.pth'

    if vname in results and ckpt_file.exists():
        ta = results[vname]['acc']
        tf = results[vname]['f1']
        print(f'[SKIP] {vname}  (acc={ta:.4f}  f1={tf:.4f}) — checkpoint found')
        continue

    epoch_ckpts = sorted(_glob.glob(str(CKPT_DIR / f'{sname}_epoch*.pth')))
    resume_ckpt = epoch_ckpts[-1] if epoch_ckpts else None
    if resume_ckpt:
        print(f'[RESUME] {vname}  — found partial checkpoint: {resume_ckpt}')

    print(f'\n{"="*64}')
    print(f'  {vname}')
    print(f'  gate={pos}  png={png}  epochs={epochs}  lr={lr}')
    print(f'{"="*64}')

    try:
        model = build_model(pos=pos, png=png)
        log   = train(model, epochs=epochs, lr=lr, name=vname, resume_ckpt=resume_ckpt)

        test_metrics = evaluate(model, test_dl)
        results[vname]    = test_metrics
        train_logs[vname] = log

        torch.save(model.state_dict(), ckpt_file)

        with open(RESULTS_FILE, 'w') as _f:
            _json.dump(results, _f, indent=2)

        ta = test_metrics['acc']
        tf = test_metrics['f1']
        print(f'  TEST  acc={ta:.4f}  f1={tf:.4f}')
        print(f'  Saved checkpoint  -> {ckpt_file}')
        print(f'  Saved results.json -> {RESULTS_FILE}')

    except Exception as _e:
        print(f'  [ERROR] {vname} failed: {_e}')
        _tb.print_exc()
        print('  Skipping — continuing to next experiment...')

    finally:
        try:
            del model
        except NameError:
            pass
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

n_done = len(results)
n_total = len(EXPERIMENTS)
print(f'\nDone. {n_done}/{n_total} experiments completed.')

In [ ]:
# ── Results table ─────────────────────────────────────────────────────────────
import json as _json
if not results and (CKPT_DIR / 'results.json').exists():
    with open(CKPT_DIR / 'results.json') as _f:
        results = _json.load(_f)
    print('Loaded results from results.json')

rows = []
for vname, m in results.items():
    rows.append({
        'Variant':    vname,
        'Acc':        round(m['acc'], 4),
        'F1 (macro)': round(m['f1'],  4),
        'Precision':  round(m['precision'], 4),
        'Recall':     round(m['recall'],    4),
    })

df_results = pd.DataFrame(rows).set_index('Variant')
display(df_results)

## 5. Figures

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig A — Bar chart: Accuracy & F1 across all 7 variants
# ─────────────────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', font_scale=1.1)

names  = list(results.keys())
accs   = [results[n]['acc'] for n in names]
f1s    = [results[n]['f1']  for n in names]
colors = ['#e15759' if 'PNG' in n else '#4e79a7' for n in names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fig A  |  Gate Position Ablation — Offensive Language Detection (Unfrozen)',
             fontsize=14, fontweight='bold', y=1.01)

for ax, vals, title, ylbl in zip(
    axes,
    [accs, f1s],
    ['Test Accuracy', 'Test F1 (macro)'],
    ['Accuracy', 'F1'],
):
    bars = ax.bar(range(len(names)), vals, color=colors,
                  edgecolor='white', linewidth=0.6)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(ylbl, fontsize=12)
    ax.set_title(title, fontsize=12)
    lo = min(vals) - 0.03
    hi = max(vals) + 0.03
    ax.set_ylim(lo, hi)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.003,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4e79a7', label='Gate variants'),
    Patch(facecolor='#e15759', label='PNG (novel)'),
]
axes[1].legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('fig_A_ablation_bar.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_A_ablation_bar.pdf')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig B — Training curves: Baseline vs G1 vs PNG  (val loss + val F1)
# ─────────────────────────────────────────────────────────────────────────────
highlight = ['Baseline', 'G1 - Attn Output', 'PNG (novel)']
palette   = {
    'Baseline':          '#4e79a7',
    'G1 - Attn Output':  '#59a14f',
    'PNG (novel)':       '#e15759',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fig B  |  Training Curves — Baseline vs G1 vs PNG (Unfrozen)',
             fontsize=14, fontweight='bold', y=1.01)

for ax, log_key, ylabel in zip(
    axes,
    ['val_loss', 'val_f1'],
    ['Val Loss', 'Val F1 (macro)'],
):
    for vname in highlight:
        if vname in train_logs:
            vals = train_logs[vname][log_key]
            ax.plot(
                range(1, len(vals) + 1), vals,
                label=vname, color=palette[vname],
                linewidth=2, marker='o', markersize=5,
            )
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xticks(range(1, EPOCHS + 1))

plt.tight_layout()
plt.savefig('fig_B_training_curves.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_B_training_curves.pdf')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig C — Layer x Head gate activation heatmap (PNG model)
# ─────────────────────────────────────────────────────────────────────────────
print('Rebuilding PNG model for gate visualisation...')
m_png = build_model(pos='PNG', png=True)

safe_png = _safe_name('PNG (novel)')
ckpt_path = CKPT_DIR / f'{safe_png}.pth'
if ckpt_path.exists():
    m_png.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=False))
    print('Loaded PNG checkpoint')
else:
    print('Checkpoint not found — using random gate params')

attn_modules = [mod for mod in m_png.modules() if _is_bert_self_attn(mod)]
for mod in attn_modules:
    mod._capture_gate = True

m_png.eval()
sample_batch = next(iter(val_dl))
with torch.no_grad():
    _ = m_png(
        input_ids=sample_batch['input_ids'].to(DEVICE),
        attention_mask=sample_batch['attention_mask'].to(DEVICE),
    )

n_layers = len(attn_modules)
n_heads  = attn_modules[0]._last_gate.shape[-1]
layer_head_gate = np.zeros((n_layers, n_heads))
for i, mod in enumerate(attn_modules):
    gate = mod._last_gate.numpy()
    layer_head_gate[i] = gate.mean(axis=(0, 1))

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    layer_head_gate,
    annot=True, fmt='.2f', cmap='RdYlGn',
    xticklabels=[f'H{i+1}' for i in range(n_heads)],
    yticklabels=[f'L{i+1}' for i in range(n_layers)],
    vmin=0.0, vmax=1.0, ax=ax,
)
ax.set_xlabel('Attention Head', fontsize=12)
ax.set_ylabel('Transformer Layer', fontsize=12)
ax.set_title(
    'Fig C  |  PNG Gate Activations — Mean over Batch & Tokens (Layer x Head) [Unfrozen]',
    fontsize=13, fontweight='bold',
)
plt.tight_layout()
plt.savefig('fig_C_gate_heatmap.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_C_gate_heatmap.pdf')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig D — Scatter: gate suppression vs token norm
# ─────────────────────────────────────────────────────────────────────────────
all_norms, all_gates = [], []

for mod in attn_modules:
    gate  = mod._last_gate.numpy()
    xnorm = mod._last_x_norm.numpy()
    mean_gate = gate.mean(axis=-1)
    all_norms.append(xnorm.flatten())
    all_gates.append(mean_gate.flatten())

all_norms = np.concatenate(all_norms)
all_gates = np.concatenate(all_gates)

rng = np.random.default_rng(SEED)
idx = rng.choice(len(all_norms), size=min(3000, len(all_norms)), replace=False)
x_plot, y_plot = all_norms[idx], all_gates[idx]

r, p_val = pearsonr(x_plot, y_plot)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_plot, y_plot, alpha=0.25, s=8, c='#e15759', rasterized=True)
ax.set_xlabel('Token Hidden-State Norm  ||h||_2', fontsize=12)
ax.set_ylabel('PNG Gate Activation  G  (mean over heads)', fontsize=12)
ax.set_title(
    f'Fig D  |  PNG Gate Suppression vs Token Norm  (r={r:.3f}, p={p_val:.2e}) [Unfrozen]',
    fontsize=13, fontweight='bold',
)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='G=0.5 (neutral)')
ax.legend()
plt.tight_layout()
plt.savefig('fig_D_gate_vs_norm.pdf', bbox_inches='tight')
plt.show()
print(f'Saved fig_D_gate_vs_norm.pdf  (Pearson r={r:.3f})')
del m_png

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig E — Per-class F1 breakdown: Baseline vs G1 vs PNG
# ─────────────────────────────────────────────────────────────────────────────
top_variants = [
    ('Baseline',          'baseline', False),
    ('G1 - Attn Output',  'G1',       False),
    ('PNG (novel)',        'PNG',      True),
]
per_class_f1 = {}

for vname, pos, png in top_variants:
    m = build_model(pos=pos, png=png)
    safe = _safe_name(vname)
    ckpt = CKPT_DIR / f'{safe}.pth'
    if ckpt.exists():
        m.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=False))
    m.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_dl:
            ids  = batch['input_ids'].to(DEVICE)
            attn = batch['attention_mask'].to(DEVICE)
            out  = m(input_ids=ids, attention_mask=attn)
            all_preds.extend(out.logits.argmax(-1).cpu().numpy())
            all_labels.extend(batch['label'].numpy())
    per_class_f1[vname] = f1_score(all_labels, all_preds, average=None, zero_division=0)
    print(f'{vname}: {per_class_f1[vname]}')
    del m

n_classes = len(LABEL_NAMES)
x     = np.arange(n_classes)
width = 0.25
bar_colors = ['#4e79a7', '#59a14f', '#e15759']

fig, ax = plt.subplots(figsize=(8, 5))
for i, (vname, _, _) in enumerate(top_variants):
    f1_cls = per_class_f1[vname]
    bars = ax.bar(
        x + (i - 1) * width, f1_cls,
        width, label=vname, color=bar_colors[i], edgecolor='white',
    )
    for bar, v in zip(bars, f1_cls):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(LABEL_NAMES, fontsize=11)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_title('Fig E  |  Per-Class F1 — Baseline vs G1 vs PNG (Unfrozen)',
             fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('fig_E_per_class_f1.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_E_per_class_f1.pdf')

## Fig F — Token Attention Heatmap (BERT → Words)
Analogous to ViT patch-attention maps: shows which **words** each model attends to for individual examples.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig F — Token-level Attention Heatmap  (BERT words  ←→  ViT patches)
# ─────────────────────────────────────────────────────────────────────────────

SPECIAL_TOKENS = {'[CLS]', '[SEP]', '[PAD]'}

def _capture_forward(mod, captured, layer_idx):
    import math as _math
    import types as _types
    import torch.nn.functional as _F

    def _fwd(self, hidden_states, attention_mask=None,
             head_mask=None, encoder_hidden_states=None,
             encoder_attention_mask=None, past_key_value=None,
             output_attentions=False, **kwargs):
        bs, q_len, _ = hidden_states.size()
        H  = self.num_attention_heads
        dh = self.attention_head_size

        def _t(x):
            return x.view(bs, q_len, H, dh).permute(0, 2, 1, 3)

        q = _t(self.query(hidden_states))
        k = _t(self.key(hidden_states))
        v = _t(self.value(hidden_states))

        scores = torch.matmul(q, k.transpose(-1, -2)) / _math.sqrt(dh)
        if attention_mask is not None:
            scores = scores + attention_mask
        attn_probs = _F.softmax(scores, dim=-1)
        attn_probs = self.dropout(attn_probs)
        if head_mask is not None:
            attn_probs = attn_probs * head_mask

        captured[layer_idx] = attn_probs[0].mean(dim=0)[0].detach().cpu().numpy()

        context = torch.matmul(attn_probs, v)
        context = context.permute(0, 2, 1, 3).contiguous()
        context = context.view(bs, q_len, self.all_head_size)
        return (context, attn_probs)

    return _types.MethodType(_fwd, mod)


def get_cls_attention(model, input_ids, attention_mask):
    model.eval()
    attn_mods = [mod for mod in model.modules() if _is_bert_self_attn(mod)]

    if hasattr(model, 'gate_params'):
        for mod in attn_mods:
            mod._capture_attn = True
        with torch.no_grad():
            _ = model(input_ids=input_ids, attention_mask=attention_mask)
        attn_layers = np.stack(
            [mod._last_attn[0].mean(dim=0)[0].cpu().numpy() for mod in attn_mods]
        )
        for mod in attn_mods:
            mod._capture_attn = False
    else:
        captured = {}
        saved_forwards = {i: mod.forward for i, mod in enumerate(attn_mods)}
        for i, mod in enumerate(attn_mods):
            mod.forward = _capture_forward(mod, captured, i)
        with torch.no_grad():
            _ = model(input_ids=input_ids, attention_mask=attention_mask)
        for i, mod in enumerate(attn_mods):
            mod.forward = saved_forwards[i]
        attn_layers = np.stack([captured[i] for i in range(len(attn_mods))])

    return attn_layers, attn_layers.mean(axis=0)


def decode_tokens(input_ids_1d):
    toks = tokenizer.convert_ids_to_tokens(input_ids_1d.tolist())
    return [t for t in toks if t != '[PAD]']


def strip_and_renorm(toks, attn):
    keep = [i for i, t in enumerate(toks) if t not in SPECIAL_TOKENS]
    toks_c = [toks[i] for i in keep]
    attn_c = attn[keep]
    total  = attn_c.sum()
    if total > 1e-9:
        attn_c = attn_c / total
    return toks_c, attn_c


EXAMPLE_INDICES = []
test_raw = raw['test']
targets  = [1, 1, 0, 0]

for label_want in targets:
    for i, ex in enumerate(test_raw):
        if ex['label'] == label_want and i not in EXAMPLE_INDICES:
            EXAMPLE_INDICES.append(i)
            break

print('Selected examples:')
for idx in EXAMPLE_INDICES:
    ex = test_raw[idx]
    print(f'  [{idx}] label={LABEL_NAMES[ex["label"]]:15s}  "{ex["text"][:80]}"')


VARIANTS_F = [
    ('Baseline',         'baseline', False),
    ('G1 - Attn Output', 'G1',       False),
    ('PNG (novel)',       'PNG',      True),
]

models_f = {}
for vname, pos, png in VARIANTS_F:
    m = build_model(pos=pos, png=png)
    ckpt = CKPT_DIR / f'{_safe_name(vname)}.pth'
    if ckpt.exists():
        m.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=False))
        print(f'Loaded checkpoint for {vname}')
    else:
        print(f'[WARN] No checkpoint for {vname} -- using random weights')
    m.eval()
    models_f[vname] = m


attn_maps = {v: {} for v, _, _ in VARIANTS_F}

for idx in EXAMPLE_INDICES:
    enc = tokenizer(
        test_raw[idx]['text'],
        truncation=True, max_length=MAX_LEN,
        padding='max_length', return_tensors='pt',
    )
    ids  = enc['input_ids'].to(DEVICE)
    mask = enc['attention_mask'].to(DEVICE)
    raw_toks = decode_tokens(enc['input_ids'][0])

    for vname, _, _ in VARIANTS_F:
        _, mean_attn = get_cls_attention(models_f[vname], ids, mask)
        toks_c, attn_c = strip_and_renorm(raw_toks, mean_attn[:len(raw_toks)])
        attn_maps[vname][idx] = (toks_c, attn_c)


n_examples = len(EXAMPLE_INDICES)
n_variants = len(VARIANTS_F)

fig, axes = plt.subplots(
    n_examples, n_variants,
    figsize=(6 * n_variants, 2.8 * n_examples),
    constrained_layout=True,
)
fig.suptitle(
    'Fig F  |  BERT Token Attention Heatmap — CLS->Word [Unfrozen]\n'
    '(special tokens removed, renormalised to content words)',
    fontsize=13, fontweight='bold',
)

for row, idx in enumerate(EXAMPLE_INDICES):
    ex    = test_raw[idx]
    label = LABEL_NAMES[ex['label']]
    color = '#e15759' if ex['label'] == 1 else '#4e79a7'

    for col, (vname, _, _) in enumerate(VARIANTS_F):
        ax = axes[row, col]
        toks, attn = attn_maps[vname][idx]

        sns.heatmap(
            attn[np.newaxis, :],
            ax=ax,
            cmap='YlOrRd',
            xticklabels=toks,
            yticklabels=[''],
            cbar=(col == n_variants - 1),
            linewidths=0.3,
            linecolor='white',
        )
        ax.set_xticklabels(toks, rotation=45, ha='right', fontsize=7.5)
        if row == 0:
            ax.set_title(vname, fontsize=11, fontweight='bold')
        if col == 0:
            ax.set_ylabel(
                f'[{label}]',
                fontsize=9, color=color, fontweight='bold', rotation=0,
                labelpad=55, va='center',
            )

plt.savefig('fig_F_token_attention_heatmap.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_F_token_attention_heatmap.pdf')

for m in models_f.values():
    del m
if DEVICE == 'cuda':
    torch.cuda.empty_cache()